### 1. Imports & Setup

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import mutual_info_regression, mutual_info_classif

import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

### 2. Load Data

In [2]:
df = pd.read_csv('../data/processed/processed_data.csv')

df.head()

,UserID,Email,RegistrationDate,LastActive,Age,Gender,SkinType,SkinConcerns,FitzpatrickScale,Allergies,...,SkinType_encoded,Gender_encoded,SkinConcerns_encoded,AgeGroup_encoded,BudgetTier_encoded,Age_scaled,MonthlyBudget_USD_scaled,ProductEffectiveness_Score_scaled,CustomerSatisfaction_pct_scaled,ActiveIngredientsCount_scaled
0,DERMAI_000000,user1825@gmail.com,2026-04-18 14:54:55.322730,2026-05-05 14:54:55.763338,39.0,Female,Normal,Pigmentation,4,NaN,...,106,20,140,2,1,0.448226,-0.952357,0.887376,0.785833,-0.643655
1,DERMAI_000001,user4507@gmail.com,2024-08-21 14:54:55.322730,2026-05-13 14:54:55.763338,32.0,Male,Combination,Pigmentation,5,Essential Oils,...,35,100,140,1,1,-0.188066,-0.952357,-2.412521,-1.278152,-1.398198
2,DERMAI_000002,user3658@gmail.com,2024-12-11 14:54:55.322730,2026-04-29 14:54:55.763338,41.0,Female,Oily,Pigmentation,5,NaN,...,145,20,140,2,2,0.630024,0.341663,-0.154697,0.269837,-1.398198
3,DERMAI_000003,user1680@hotmail.com,2025-09-21 14:54:55.322730,2026-04-19 14:54:55.763338,52.0,Male,Dry,Pigmentation,1,NaN,...,91,100,140,3,2,1.629912,0.341663,-1.196770,0.166637,-0.643655
4,DERMAI_000004,user8936@gmail.com,2024-07-13 14:54:55.322730,2026-04-26 14:54:55.763338,31.0,Female,Oily,Dryness,4,NaN,...,145,20,68,1,1,-0.278965,-0.952357,-2.065164,-0.658957,-0.643655


### 3. Train/Test Split

In [3]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

train_df.shape
test_df.shape

(20028, 91)

### 4. Feature Engineering Function

In [4]:
def feature_engineering(data):

    df = data.copy()

    # -------------------------
    # Interaction Features
    # -------------------------
    df['Age_Budget_Interaction'] = df['Age'] * df['MonthlyBudget_USD'] / 100
    df['Effectiveness_Satisfaction'] = df['ProductEffectiveness_Score'] * df['CustomerSatisfaction_pct'] / 100
    df['Budget_Effectiveness_Ratio'] = df['MonthlyBudget_USD'] / (df['ProductEffectiveness_Score'] + 0.1)

    # -------------------------
    # Polynomial Features
    # -------------------------
    df['Age_Squared'] = df['Age'] ** 2
    df['Budget_Squared'] = df['MonthlyBudget_USD'] ** 2 / 1000
    df['Age_Cuberoot'] = np.cbrt(df['Age'])
    df['Log_Budget'] = np.log1p(df['MonthlyBudget_USD'])

    # -------------------------
    # Aggregated Features
    # -------------------------
    ingredient_cols = [c for c in df.columns if c.startswith('Uses')]
    if ingredient_cols:
        df['Total_Active_Ingredients'] = df[ingredient_cols].sum(axis=1)
        df['Ingredient_Diversity'] = df[ingredient_cols].mean(axis=1) * 100

    # Premium score
    premium_cols = [c for c in ['UsesRetinol','UsesPeptides','UsesCeramides'] if c in df.columns]
    if premium_cols:
        df['Premium_Ingredient_Score'] = df[premium_cols].mean(axis=1) * 100

    # -------------------------
    # Routine Score
    # -------------------------
    routine_cols = [c for c in ['SleepHours','StressLevel_1to10','WaterIntake_Liters','Exercise_Weekly'] if c in df.columns]

    if routine_cols:
        scaler = MinMaxScaler()
        normalized = scaler.fit_transform(df[routine_cols])
        df['Routine_Consistency'] = normalized.mean(axis=1) * 100

    # -------------------------
    # Ratios
    # -------------------------
    df['Effectiveness_per_Dollar'] = df['ProductEffectiveness_Score'] / (df['MonthlyBudget_USD'] + 1)
    df['Satisfaction_per_Ingredient'] = df['CustomerSatisfaction_pct'] / (df.get('Total_Active_Ingredients', 1) + 1)
    df['Age_Budget_Ratio'] = df['Age'] / (df['MonthlyBudget_USD'] + 1)

    if 'StressLevel_1to10' in df.columns and 'SleepHours' in df.columns:
        df['Stress_Sleep_Ratio'] = df['StressLevel_1to10'] / (df['SleepHours'] + 0.1)

    # -------------------------
    # Categorical Features
    # -------------------------
    df['Age_Category'] = pd.cut(df['Age'],
                                bins=[18,25,35,45,55,70],
                                labels=['18-24','25-34','35-44','45-54','55+'])

    df['Budget_Category'] = pd.cut(df['MonthlyBudget_USD'],
                                   bins=[0,50,100,200,500,1000],
                                   labels=['Low','Mid','High','Premium','Luxury'])

    # -------------------------
    # Domain Features
    # -------------------------
    concern_map = {
        'Acne':5,'Pigmentation':4,'Wrinkles':4,
        'Redness':3,'Dryness':3,'Large Pores':2,'Dullness':1
    }

    if 'SkinConcerns' in df.columns:
        df['SkinConcerns'] = df['SkinConcerns'].astype(str).str.title()
        df['Concern_Severity'] = df['SkinConcerns'].map(concern_map).fillna(3)

    climate_map = {
        'Tropical':5,'Dry':4,'Mediterranean':3,
        'Temperate':3,'Continental':2,'Cold':1
    }

    if 'Climate' in df.columns:
        df['Climate_Score'] = df['Climate'].map(climate_map).fillna(3)

    return df

### 5. Apply Feature Engineering

In [5]:
train_df = feature_engineering(train_df)
test_df = feature_engineering(test_df)

print("Feature engineering completed safely")

Feature engineering completed safely


### 6. Safe Scaling

In [6]:
scale_cols = [
    'Age','MonthlyBudget_USD',
    'ProductEffectiveness_Score',
    'CustomerSatisfaction_pct',
    'Total_Active_Ingredients',
    'Routine_Consistency',
    'Age_Budget_Interaction',
    'Effectiveness_Satisfaction'
]

scale_cols = [c for c in scale_cols if c in train_df.columns]

scaler = StandardScaler()

train_df[scale_cols] = scaler.fit_transform(train_df[scale_cols])
test_df[scale_cols] = scaler.transform(test_df[scale_cols])

joblib.dump(scaler, '../data/models/feature_scaler.pkl')

['../data/models/feature_scaler.pkl']

### 7. Feature Importance

In [7]:
def get_mi_scores(X, y):
    if y.nunique() <= 10:
        return mutual_info_classif(X, y, random_state=42)
    else:
        return mutual_info_regression(X, y, random_state=42)

target_cols = ['ProductEffectiveness_Score','CustomerSatisfaction_pct','WillRepurchase']

numerical_cols = train_df.select_dtypes(include=[np.number]).columns

feature_importance = {}

for target in target_cols:
    if target in train_df.columns:

        X = train_df[numerical_cols].drop(columns=target_cols, errors='ignore')
        y = train_df[target]

        X = X.fillna(0)

        mi = get_mi_scores(X, y)

        fi = pd.DataFrame({
            'Feature': X.columns,
            'Importance': mi
        }).sort_values(by='Importance', ascending=False)

        feature_importance[target] = fi

        print("\nTop features for:", target)
        print(fi.head(10))


Top features for: ProductEffectiveness_Score
                              Feature  Importance
66           Effectiveness_per_Dollar    3.102436
52  ProductEffectiveness_Score_scaled    3.101142
57         Budget_Effectiveness_Ratio    3.002917
56         Effectiveness_Satisfaction    2.644699
43                       OverallScore    0.493031
47               SkinConcerns_encoded    0.007559
58                        Age_Squared    0.006414
0                                 Age    0.005820
69                 Stress_Sleep_Ratio    0.005065
29               ValueForMoney_Rating    0.004480

Top features for: CustomerSatisfaction_pct
                            Feature  Importance
53  CustomerSatisfaction_pct_scaled    3.631239
67      Satisfaction_per_Ingredient    3.502978
56       Effectiveness_Satisfaction    3.170716
43                     OverallScore    1.019770
41           PregnancyBreastfeeding    0.008140
63             Ingredient_Diversity    0.006451
21                     U